In [13]:
# Task 5: Per-Tensor and Per-Channel Quantization

## Objective

#The objective of this task is to compare per-tensor and per-output-channel symmetric quantization for convolution weight tensors.

#In per-tensor quantization, a single scale is used for the entire weight tensor. In per-channel quantization, each output channel has its own scale. The reconstructed tensors are compared using Mean Absolute Error (MAE) to determine which method preserves the original weights more accurately.

import numpy as np

# Generate convolution weight tensor

np.random.seed(42)

conv_weights = np.random.randn(8, 3, 3, 3).astype(np.float32)

# Create different value ranges
conv_weights[0] *= 0.1
conv_weights[1] *= 0.5
conv_weights[6] *= 5.0
conv_weights[7] *= 10.0

print("Tensor Shape:", conv_weights.shape)

# Per-Tensor Quantization

def per_tensor_quantize(tensor):

    max_abs = np.max(np.abs(tensor))

    scale = max_abs / 127

    quantized = np.round(tensor / scale)

    quantized = np.clip(quantized, -127, 127)

    quantized = quantized.astype(np.int8)

    dequantized = quantized.astype(np.float32) * scale

    return quantized, dequantized, scale

# Per-Channel Quantization

def per_channel_quantize(tensor):

    quantized = np.zeros_like(tensor, dtype=np.int8)

    dequantized = np.zeros_like(tensor, dtype=np.float32)

    scales = []

    for i in range(tensor.shape[0]):

        max_abs = np.max(np.abs(tensor[i]))

        if max_abs == 0:
            scale = 1.0
        else:
            scale = max_abs / 127

        scales.append(scale)

        q = np.round(tensor[i] / scale)

        q = np.clip(q, -127, 127)

        quantized[i] = q.astype(np.int8)

        dequantized[i] = quantized[i].astype(np.float32) * scale

    return quantized, dequantized, np.array(scales)

# Apply Quantization

pt_quantized, pt_dequantized, pt_scale = per_tensor_quantize(conv_weights)

pc_quantized, pc_dequantized, pc_scales = per_channel_quantize(conv_weights)

# Compare Both Methods

pt_mae_list = []
pc_mae_list = []

for i in range(conv_weights.shape[0]):

    original = conv_weights[i]

    pt_mae = np.mean(np.abs(original - pt_dequantized[i]))

    pc_mae = np.mean(np.abs(original - pc_dequantized[i]))

    pt_mae_list.append(pt_mae)

    pc_mae_list.append(pc_mae)

    print("\n===================================")
    print("Channel :", i)
    print("-----------------------------------")

    print("Minimum Value :", np.min(original))
    print("Maximum Value :", np.max(original))

    print()

    print("Per-Tensor Scale :", pt_scale)
    print("Per-Tensor MAE   :", pt_mae)

    print()

    print("Per-Channel Scale :", pc_scales[i])
    print("Per-Channel MAE   :", pc_mae)

    print()

    if pc_mae < pt_mae:
        print("Better Method : Per-Channel")
    else:
        print("Better Method : Per-Tensor")

print("\n===================================")
print("Average Per-Tensor MAE :", np.mean(pt_mae_list))
print("Average Per-Channel MAE :", np.mean(pc_mae_list))

print("\n\nSummary Table")

print("="*95)

print("Channel\tPT MAE\t\tPC MAE\t\tBetter")

print("="*95)

for i in range(8):

    pt_mae = np.mean(np.abs(conv_weights[i] - pt_dequantized[i]))
    pc_mae = np.mean(np.abs(conv_weights[i] - pc_dequantized[i]))

    if pc_mae < pt_mae:
        better = "Per-Channel"
    else:
        better = "Per-Tensor"

    print(i,"\t",
          round(pt_mae,6),"\t",
          round(pc_mae,6),"\t",
          better)

print("="*95)


# Analysis Questions

### 1. Why does per-channel quantization usually produce lower error?

#Per-channel quantization assigns a separate scale to each output channel. This allows each channel to use a scale that matches its own value range, reducing the reconstruction error.

#---

### 2. Which channels benefit the most from per-channel quantization?

#Channels with very small or very large value ranges benefit the most because they no longer share one common scale with all other channels.

##---

### 3. Why was range imbalance introduced into the weight tensor?

#The imbalance was introduced to simulate real convolution layers, where different filters often learn weights with different numerical ranges.

#---

### 4. Which quantization method would you recommend for convolution weights?

#Per-channel quantization is recommended because it preserves the original weights more accurately and generally results in lower quantization error.



# Observations

#- Per-tensor quantization uses one common scale for all output channels.
#- Per-channel quantization computes a separate scale for every output channel.
#- Channels with smaller value ranges experience higher error when using per-tensor quantization because the common scale is influenced by channels with much larger values.
#- Per-channel quantization reduces the reconstruction error for most channels.
#- The average MAE is lower for per-channel quantization, indicating better preservation of the original weights.

# Conclusion

#In this task, both per-tensor and per-channel symmetric quantization were implemented for convolution weight tensors. The comparison showed that per-channel quantization generally produces lower reconstruction error because each output channel has its own quantization scale. This makes per-channel quantization the preferred choice for convolutional neural networks, especially when different channels have significantly different value ranges.



Tensor Shape: (8, 3, 3, 3)

Channel : 0
-----------------------------------
Minimum Value : -0.19132803
Maximum Value : 0.15792128

Per-Tensor Scale : 0.30336466
Per-Tensor MAE   : 0.07146436

Per-Channel Scale : 0.00150652
Per-Channel MAE   : 0.00032642912

Better Method : Per-Channel

Channel : 1
-----------------------------------
Minimum Value : -0.97983503
Maximum Value : 0.9261391

Per-Tensor Scale : 0.30336466
Per-Tensor MAE   : 0.07256411

Per-Channel Scale : 0.0077152364
Per-Channel MAE   : 0.0016606675

Better Method : Per-Channel

Channel : 2
-----------------------------------
Minimum Value : -2.619745
Maximum Value : 1.5646436

Per-Tensor Scale : 0.30336466
Per-Tensor MAE   : 0.072174944

Per-Channel Scale : 0.020627914
Per-Channel MAE   : 0.005372616

Better Method : Per-Channel

Channel : 3
-----------------------------------
Minimum Value : -1.4635149
Maximum Value : 1.8861859

Per-Tensor Scale : 0.30336466
Per-Tensor MAE   : 0.071593724

Per-Channel Scale : 0.014851857